# Phase 8 Consolidated Report

This notebook loads Phase 7 summaries and builds the narrative comparing diagnostics, congruence, multiblock, and baseline results.


In [ ]:
import json
from pathlib import Path
import pandas as pd

manifest = json.loads(Path('analysis/phase8/inputs/phase8_input_manifest.json').read_text())
modality_summary = pd.read_csv(manifest['phase7']['modality_summary']['path'])
kind_modality_summary = pd.read_csv(manifest['phase7']['kind_modality_summary']['path'])
baseline_summary = pd.read_csv(manifest['phase7']['baseline_summary']['path'])
baseline_loocv = pd.read_csv(manifest['phase7']['baseline_loocv']['path'])
modality_summary.head(), kind_modality_summary.head()


In [ ]:
comparison_overview = pd.read_csv('analysis/phase8/tables/comparison_overview.csv')
comparison_overview.head()


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
comparison_overview.pivot(index='kind', columns='modality', values='baseline_rmse_loocv').plot(kind='bar', ax=ax)
ax.set_ylabel('Baseline RMSE (LOOCV)')
ax.set_title('LOOCV RMSE by Kind/Modality (Phase 9 ridge)')
ax.legend(title='Modality')
plt.tight_layout()
plt.show()


## Comparison Overview

The table above merges Phase 4 dependence metrics, Phase 5 multiblock components, and Phase 9 ridge baseline LOOCV errors for quick reference.


In [ ]:
summary_stats = (comparison_overview.groupby('modality')[['dcor_permutation_p', 'rv_permutation_p', 'baseline_rmse_loocv', 'baseline_mae_loocv']]
                   .agg(['min', 'max'])
                   .round(3))
summary_stats


In [ ]:
lowest_rmse = (comparison_overview.sort_values('baseline_rmse_loocv')
                [['modality', 'kind', 'baseline_rmse_loocv', 'dcor_permutation_p', 'rv_permutation_p']]
                .head(10))
lowest_rmse


### Interpretation

- Even the lowest distance-correlation or RV permutation p-values remain well above 0.05, so none of the reflectance–concentration alignments reach conventional significance.
- Baseline LOOCV RMSEs stay between ~1.6 and ~3.7 depending on kind/modality, much higher than the in-sample RMSE reported in Phase 9 ridge (≈1e-12). This reinforces the overfitting warning.
- Multiblock runs only a single joint component per modality with no warnings, mirroring the modest dependence signal.
- 12Oclock measurements consistently give the lowest LOOCV RMSE among the band baselines; Δ and 6 o'clock remain the noisiest.
- Taken together, the analytics suggest strong in-sample coherence but limited predictive power without more samples or regularisation.
